In [ ]:
import qewton

path = "/localdata/tomfre/SolutionData/MinimalSurface/"

X = qewton.Variable("x", 2)
U = qewton.Variable("u", 1) 
N = qewton.Variable("n", 2)

In [ ]:
omega = qewton.geometries.MeshGeometry.load_mesh(
    variable=X, 
    file_path=path + "unit_square_minus_circle.msh",
    prune_z=True
)

gamma_D = omega.boundary.get_submesh("dirichlet")
gamma_N = omega.boundary.get_submesh("neumann")

In [ ]:
qewton.viz.Figure(qewton.viz.GeometryPlot(omega)).show()

In [ ]:
inner_sampler = qewton.GridSampler(omega, n_points=25000)
gamma_D_sampler = qewton.GridSampler(gamma_D, n_points=8000)
gamma_N_sampler = qewton.GridSampler(
    gamma_N, n_points=8000, normal_name=N, compute_normals=True)

In [ ]:
model = qewton.FCN(in_neurons=X, 
                   out_neurons=U, 
                   hidden_neurons=32, 
                   n_hidden_layers=3,
                   activation=qewton.bb.Tanh)

In [ ]:
def pde_residual(u : U, x : X):
    u_grad = u.gradient(x)
    u_grad_norm = u_grad.norm()
    scaling = (1.0 + u_grad_norm**2).sqrt()
    return - (u_grad / scaling).div(x) + 1.0

pde_constraint = qewton.constraints.PINNConstraint(pde_residual)

pde_pipeline = qewton.PINNPipeline(
    sampler=inner_sampler,
    constraint=pde_constraint,
    models=[model]
)

In [ ]:
def dirichlet_residual(u : U):
    return u

dirichlet_constraint = qewton.constraints.PINNConstraint(
    dirichlet_residual, 
    weight=10.0, 
    name="DirichletConstraint"
)
dirichlet_pipeline = qewton.PINNPipeline(gamma_D_sampler, [model], dirichlet_constraint)

In [ ]:
def neumann_residual(u: U, x: X, n: N):
    return u.normal_derivative(n, x)

neuman_constraint = qewton.constraints.PINNConstraint(
    neumann_residual, 
    weight=10.0, 
    name="NeumannConstraint"
)
neumann_pipeline = qewton.PINNPipeline(gamma_N_sampler, [model], neuman_constraint)

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=2500
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    lr=1.0,
    max_iterations=100,
    optimizer_args={"max_eval": 5}
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase, lbfgs_phase],
    graphs=[pde_pipeline, dirichlet_pipeline, neumann_pipeline],
    training_objectives=[pde_constraint, dirichlet_constraint, neuman_constraint],
    device=qewton.cuda(0)
)

trainer.run()

In [ ]:
solution_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "solution.pt")
reference_config = qewton.DataConfiguration(
    qewton.GeometryAxes(geometry=omega), qewton.FeatureAxes(U)
)

layout = pde_pipeline.visualize(model.output_ports[0], 
                             reference=solution_data, 
                             reference_config=reference_config)
qewton.viz.Figure(layout).show()